# Chapter 5: Understanding MCP
## Architecture and Implementation - Code Examples

This notebook contains the runnable code from Chapter 5. It builds three MCP servers, connects to them over both standard transports, exercises each primitive from the client side, hardens a server with validation, authorization, and retries, and finishes by putting MCP tools in front of a language model twice, once with the raw SDK and once through LangChain:
- A resource server, the connection lifecycle, and reading a resource
- A prompt server and the prompt primitive from the client side
- A tool server and the tool primitive from the client side
- The Streamable HTTP transport, with the session id from the handshake
- Input validation, role-based access control, and retries, tested through an in-memory session
- An agent loop on the OpenAI Responses API that discovers and calls MCP tools
- The same agent built with LangChain and the official MCP adapters

### Setup

The dependencies for every chapter are declared in `pyproject.toml` at the repository root. From the root, run:

```bash
uv sync --all-groups
```

Then start Jupyter with `uv run jupyter lab` and select the **Agentic AI Handbook (Python 3.13)** kernel.

This notebook calls the OpenAI API. Copy `.env.example` to `.env` at the repository root and add your `OPENAI_API_KEY` before running the cells.

### How the servers run

An MCP server is a separate process. Each server in this notebook is written to a file with a `%%writefile` cell, and the client cell that follows launches it as a subprocess over stdio, talks to it, and shuts it down when the cell ends. You never open a second terminal. The Streamable HTTP part starts one server in the background on localhost and stops it two cells later. Everything is written inside a temporary workspace created in the setup cell, so nothing in this repository is modified.

The chapter's listings use the Anthropic Messages API for the agent loop and `/data/documents` as the server's document folder. This notebook uses `gpt-5.6-luna` on the OpenAI Responses API, consistent with the earlier chapters, and passes the workspace path to each server as a command-line argument. Every other listing is reproduced as printed.

In [ ]:
# Import required libraries
import os
import sys
import json
import re
import time
import socket
import sqlite3
import logging
import asyncio
import tempfile
import subprocess
from contextlib import AsyncExitStack
from pathlib import Path
from typing import Optional
from dotenv import load_dotenv

# Load environment variables (API keys)
load_dotenv()

# Verify API keys are loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in environment"

# Every server file and every document lives in a throwaway workspace
WORKSPACE = Path(tempfile.mkdtemp(prefix="ch5-mcp-")).resolve()
DOCS = WORKSPACE / "data" / "documents"
DOCS.mkdir(parents=True)
os.chdir(WORKSPACE)

# Servers are launched with the same interpreter that runs this notebook
PYTHON = sys.executable

(DOCS / "notes.txt").write_text(
    "MCP quick reference\n"
    "The stable specification revision is 2025-11-25.\n"
    "Q4 2024 revenue by month: October 125000, November 131000, December 142000.\n"
)
(DOCS / "handbook.md").write_text(
    "# Team handbook\n\nRelease reviews happen every Thursday.\n"
    "Sales figures live in the quarterly CSV files.\n"
)
(DOCS / "2024-q4.csv").write_text(
    "Month,Date,Revenue,Units\n"
    "October 2024,2024-10-01,125000,450\n"
    "November 2024,2024-11-01,131000,470\n"
    "December 2024,2024-12-01,142000,510\n"
)

print("Environment setup complete")
print(f"Workspace: {WORKSPACE}")
print(f"Documents: {sorted(p.name for p in DOCS.iterdir())}")

## Part 1: A Resource Server and the Connection Lifecycle

The chapter's resource server exposes the text, Markdown, and CSV files in a folder as MCP resources. The listing below is the chapter's code with one addition: the folder path comes from the first command-line argument so the server can point at this notebook's workspace, and a `__main__` guard starts the server when the file is run.

Writing the server to a file matters. An MCP server over stdio is a program the client launches, so it has to exist on disk.

In [ ]:
%%writefile filesystem_server.py
from mcp.server import Server
from mcp.types import Resource
import mcp.server.stdio
import os
import sys

# Create MCP server instance
server = Server("filesystem-server")

# The chapter uses /data/documents; the notebook passes its workspace path instead
BASE_PATH = sys.argv[1] if len(sys.argv) > 1 else "/data/documents"

@server.list_resources()
async def list_resources() -> list[Resource]:
    """List available file resources"""
    resources = []
    base_path = BASE_PATH

    for filename in os.listdir(base_path):
        if filename.endswith(('.txt', '.md', '.csv')):
            resources.append(Resource(
                uri=f"file://{base_path}/{filename}",
                name=filename,
                description=f"Document: {filename}",
                mimeType="text/plain"
            ))

    return resources

@server.read_resource()
async def read_resource(uri: str) -> str:
    """Read content of a specific resource"""
    # Extract file path from URI
    file_path = str(uri).replace("file://", "")

    # Read and return file content
    with open(file_path, 'r') as f:
        content = f.read()

    return content

# Run the server
async def main():
    async with mcp.server.stdio.stdio_server() as (read_stream, write_stream):
        await server.run(
            read_stream,
            write_stream,
            server.create_initialization_options()
        )

if __name__ == "__main__":
    import asyncio
    asyncio.run(main())

### The connection lifecycle from the client side

The chapter's lifecycle listing configures the server command, opens the stdio transport, creates a `ClientSession`, and calls `initialize()`. Jupyter already runs an event loop, so the cell can `await` at the top level. When the `async with` blocks end, the client closes the server's stdin and the subprocess exits, which is exactly how the specification defines shutdown over stdio.

The `initialize()` result carries what the handshake negotiated: the server's name, the protocol version both sides agreed on, and the capabilities the server declared.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Initialize connection to MCP server
server_params = StdioServerParameters(
    command=PYTHON,
    args=["filesystem_server.py", str(DOCS)]
)

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        init = await session.initialize()
        print("=== Handshake ===")
        print(f"server:           {init.serverInfo.name} {init.serverInfo.version}")
        print(f"protocol version: {init.protocolVersion}")
        print(f"capabilities:     {list(init.capabilities.model_dump(exclude_none=True))}")

        # Connection is now in ready state
        # Can now make requests to the server
        result = await session.list_resources()
        print("\n=== Available resources ===")
        for resource in result.resources:
            print(f"{resource.name:14s} {resource.mimeType:11s} {resource.uri}")

print("\nServer subprocess has exited")

### Reading a resource

The `read_sales_data()` function is the chapter's listing with the URI pointed at the workspace copy of the file. `read_resource()` returns a `ReadResourceResult`, and each entry in its `contents` list carries the URI, the MIME type, and the text. The small `stdio_session()` helper below wraps the three-line connection dance so later cells stay short; it is the same code as the previous cell.

In [ ]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def stdio_session(script: str, *args: str):
    """Launch a server script over stdio and yield an initialized ClientSession."""
    params = StdioServerParameters(command=PYTHON, args=[script, *args])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            yield session


async def read_sales_data(session: ClientSession):
    # Request to read a specific resource
    result = await session.read_resource(
        uri=f"file://{DOCS}/2024-q4.csv"
    )

    # Access the content
    for content in result.contents:
        print(f"URI: {content.uri}")
        print(f"MIME Type: {content.mimeType}")
        print(f"Content: {content.text[:100]}...")  # First 100 chars


# --- Run it ---

print("=== resources/read ===")
async with stdio_session("filesystem_server.py", str(DOCS)) as session:
    await read_sales_data(session)

## Part 2: A Prompt Server and the Prompt Primitive

The prompt server publishes the chapter's `code-review` template. One correction to the printed listing: the SDK passes the `get_prompt` handler's return value straight back to the client as the `prompts/get` result, so the handler must return a `GetPromptResult` that wraps the messages, not a bare list.

In [ ]:
%%writefile prompt_server.py
from mcp.server import Server
from mcp.types import Prompt, PromptArgument, GetPromptResult, PromptMessage, TextContent
import mcp.server.stdio

server = Server("prompt-templates-server")

@server.list_prompts()
async def list_prompts() -> list[Prompt]:
    """List available prompt templates"""
    return [
        Prompt(
            name="code-review",
            description="Perform a comprehensive code review",
            arguments=[
                PromptArgument(
                    name="language",
                    description="Programming language",
                    required=True
                ),
                PromptArgument(
                    name="file_path",
                    description="Path to file to review",
                    required=True
                )
            ]
        ),
    ]

@server.get_prompt()
async def get_prompt(name: str, arguments: dict[str, str] | None) -> GetPromptResult:
    """Generate prompt content based on template and arguments"""
    arguments = arguments or {}

    if name == "code-review":
        language = arguments.get("language")
        file_path = arguments.get("file_path")

        return GetPromptResult(
            description="Comprehensive code review",
            messages=[
                PromptMessage(
                    role="user",
                    content=TextContent(
                        type="text",
                        text=f"""Please perform a comprehensive code review of the following {language} file: {file_path}

Focus on:
1. Code quality and best practices
2. Potential bugs or security issues
3. Performance considerations
4. Readability and maintainability
5. Test coverage and edge cases

Provide specific, actionable feedback with code examples where appropriate."""
                    )
                )
            ],
        )

    raise ValueError(f"Unknown prompt: {name}")

async def main():
    async with mcp.server.stdio.stdio_server() as (read_stream, write_stream):
        await server.run(
            read_stream,
            write_stream,
            server.create_initialization_options()
        )

if __name__ == "__main__":
    import asyncio
    asyncio.run(main())

### Using a prompt from the client

`use_code_review_prompt()` is the chapter's listing. The client first lists the prompts and their declared arguments, then asks the server to render `code-review` for a Python file.

In [ ]:
async def use_code_review_prompt(session: ClientSession):
    # Get a prompt with arguments
    prompt = await session.get_prompt(
        name="code-review",
        arguments={
            "language": "python",
            "file_path": "src/main.py"
        }
    )

    return prompt.messages[0].content


# --- Run it ---

async with stdio_session("prompt_server.py") as session:
    listing = await session.list_prompts()
    print("=== prompts/list ===")
    for prompt in listing.prompts:
        args = ", ".join(a.name + ("*" if a.required else "") for a in prompt.arguments)
        print(f"{prompt.name}: {prompt.description} (arguments: {args})")

    print("\n=== prompts/get code-review ===")
    content = await use_code_review_prompt(session)
    print(content.text)

## Part 3: A Tool Server and the Tool Primitive

The tool server publishes the chapter's `web_search` tool and a second tool, `search_documents`, that greps the workspace documents. The chapter leaves `perform_web_search()` unimplemented; here it returns fixed results so the notebook runs offline, and a real deployment would call a search API in its place. Tool descriptions are written for the model, because the description is what the model reads when deciding whether to call the tool.

In [ ]:
%%writefile tool_server.py
from mcp.server import Server
from mcp.types import Tool, TextContent
import mcp.server.stdio
import json
import os
import sys

server = Server("utility-tools-server")

DOCS = sys.argv[1] if len(sys.argv) > 1 else "/data/documents"


def perform_web_search(query: str, max_results: int) -> list[dict]:
    """Stand-in for a real search API. Returns fixed results so the notebook runs offline."""
    catalog = [
        {"title": "Model Context Protocol specification 2025-11-25",
         "url": "https://modelcontextprotocol.io/specification/2025-11-25",
         "snippet": "The 2025-11-25 revision is the stable release of the protocol."},
        {"title": "One Year of MCP",
         "url": "https://blog.modelcontextprotocol.io/posts/2025-11-25-first-mcp-anniversary/",
         "snippet": "The registry holds close to two thousand entries since September 2025."},
        {"title": "Donating the Model Context Protocol",
         "url": "https://www.anthropic.com/news/donating-the-model-context-protocol-and-establishing-of-the-agentic-ai-foundation",
         "snippet": "More than 10,000 active public MCP servers and 97M+ monthly SDK downloads."},
    ]
    words = query.lower().split()
    hits = [r for r in catalog if any(w in json.dumps(r).lower() for w in words)]
    return (hits or catalog)[:max_results]


def search_documents(term: str) -> list[dict]:
    """Return every line in the documents folder that contains the term."""
    matches = []
    for filename in sorted(os.listdir(DOCS)):
        with open(os.path.join(DOCS, filename)) as f:
            for number, line in enumerate(f, 1):
                if term.lower() in line.lower():
                    matches.append({"file": filename, "line": number, "text": line.rstrip()})
    return matches


@server.list_tools()
async def list_tools() -> list[Tool]:
    """List available tools"""
    return [
        Tool(
            name="web_search",
            description="Search the web for information. Use this when you need current information or facts not in your training data.",
            inputSchema={
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to return",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        ),
        Tool(
            name="search_documents",
            description="Search the team's document folder for lines containing one keyword. Use this for questions about internal notes, handbooks, and sales figures. Search one word at a time, such as a month name.",
            inputSchema={
                "type": "object",
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "A single word or number to look for"
                    }
                },
                "required": ["term"]
            }
        ),
    ]


@server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    """Execute a tool and return results"""

    if name == "web_search":
        query = arguments["query"]
        max_results = arguments.get("max_results", 5)

        # In a real implementation, use actual search API
        # This is a simplified example
        results = perform_web_search(query, max_results)

        return [TextContent(
            type="text",
            text=json.dumps(results, indent=2)
        )]

    if name == "search_documents":
        results = search_documents(arguments["term"])
        return [TextContent(type="text", text=json.dumps(results, indent=2))]

    return [TextContent(
        type="text",
        text=f"Unknown tool: {name}"
    )]


async def main():
    async with mcp.server.stdio.stdio_server() as (read_stream, write_stream):
        await server.run(
            read_stream,
            write_stream,
            server.create_initialization_options()
        )

if __name__ == "__main__":
    import asyncio
    asyncio.run(main())

### Calling a tool from the client

`search_web()` is the chapter's listing. `call_tool()` returns a `CallToolResult`. Its `content` list holds the tool's output and its `isError` flag tells the client whether the call failed, which is the field a harness checks before feeding the result back to the model.

In [ ]:
async def search_web(session: ClientSession, query: str):
    # Model decides to invoke this tool during reasoning
    result = await session.call_tool(
        name="web_search",
        arguments={"query": query, "max_results": 5}
    )

    # Process tool results
    for content in result.content:
        if content.type == "text":
            print(f"Search results: {content.text}")

    return result


# --- Run it ---

async with stdio_session("tool_server.py", str(DOCS)) as session:
    listing = await session.list_tools()
    print("=== tools/list ===")
    for tool in listing.tools:
        print(f"{tool.name}: {tool.description[:70]}...")

    print("\n=== tools/call web_search ===")
    result = await search_web(session, "MCP specification stable release")
    print(f"\nisError={result.isError}, content items={len(result.content)}, type={result.content[0].type}")

    print("\n=== tools/call search_documents ===")
    result = await session.call_tool("search_documents", {"term": "November"})
    print(result.content[0].text)

## Part 4: The Streamable HTTP Transport

Over stdio the client owns the server process. Over Streamable HTTP the server is an independent process that many clients can share, and it exposes one endpoint that accepts POST and GET. The server below uses the SDK's `FastMCP` class, which builds the HTTP application for you; the low-level `Server` class can do the same with more wiring. It reuses `perform_web_search()` from the stdio server file.

In [ ]:
%%writefile tool_server_http.py
import json
import sys
from mcp.server.fastmcp import FastMCP

# FastMCP wraps the low-level Server and provides the Streamable HTTP app
mcp = FastMCP(
    "utility-tools-http",
    host="127.0.0.1",
    port=int(sys.argv[1]) if len(sys.argv) > 1 else 8765,
)


@mcp.tool()
def web_search(query: str, max_results: int = 5) -> str:
    """Search the web for information. Use this when you need current information."""
    from tool_server import perform_web_search
    return json.dumps(perform_web_search(query, max_results), indent=2)


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### Starting the server in the background

This cell starts the HTTP server as a background process on localhost and waits until the port accepts connections. The process keeps running across cells until the stop cell below ends it.

In [ ]:
HTTP_PORT = 8765
MCP_URL = f"http://127.0.0.1:{HTTP_PORT}/mcp"

http_server = subprocess.Popen(
    [PYTHON, "tool_server_http.py", str(HTTP_PORT)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait for the port to open
for _ in range(50):
    try:
        socket.create_connection(("127.0.0.1", HTTP_PORT), timeout=0.2).close()
        break
    except OSError:
        time.sleep(0.2)
else:
    raise RuntimeError("HTTP server did not start")

print(f"=== Streamable HTTP server running ===")
print(f"pid {http_server.pid}, endpoint {MCP_URL}")

### Connecting over HTTP

The SDK's HTTP client yields a third value alongside the read and write streams: a function that returns the `Mcp-Session-Id` the server assigned during initialization. The client echoes that header on every later request, which is how the server ties the requests of one session together.

In [ ]:
from mcp.client.streamable_http import streamablehttp_client

async with streamablehttp_client(MCP_URL) as (read, write, get_session_id):
    async with ClientSession(read, write) as session:
        init = await session.initialize()
        print("=== Handshake over Streamable HTTP ===")
        print(f"server:     {init.serverInfo.name}")
        print(f"session id: {get_session_id()}")

        listing = await session.list_tools()
        print(f"tools:      {[t.name for t in listing.tools]}")

        result = await session.call_tool("web_search", {"query": "MCP registry", "max_results": 1})
        print("\n=== tools/call over HTTP ===")
        print(result.content[0].text)

### Stopping the server

Shutdown over HTTP is closing the connection, which the client did when its `async with` block ended. The server process itself is ours to stop.

In [ ]:
http_server.terminate()
http_server.wait(timeout=5)
print(f"HTTP server stopped with return code {http_server.returncode}")

## Part 5: Production Patterns - Validation, Authorization, and Retries

The chapter's production section adds three layers to a tool server: input validation, role-based access control, and retries for transient failures. The SDK offers a third way to reach a server that suits testing these handlers: `create_connected_server_and_client_session()` wires a `Server` object to a `ClientSession` in memory, with no subprocess and no network. The handlers run exactly as they would behind stdio or HTTP.

First, the chapter's `validate_file_path()` with the allowed directory pointed at the workspace, and the tests from the chapter's discussion.

In [ ]:
def validate_file_path(path: str) -> str:
    # Remove any directory traversal attempts
    if ".." in path or path.startswith("/"):
        raise ValueError("Invalid file path: directory traversal not allowed")
    # Ensure path only contains safe characters
    if not re.match(r'^[a-zA-Z0-9_/\-\.]+$', path):
        raise ValueError("Invalid file path: contains unsafe characters")
    # Restrict to allowed directory
    allowed_base = str(DOCS)
    full_path = f"{allowed_base}/{path}"
    # Verify the resolved path is still within allowed directory
    real_path = os.path.realpath(full_path)
    if not real_path.startswith(os.path.realpath(allowed_base)):
        raise ValueError("Invalid file path: outside allowed directory")

    return real_path


# --- Run it ---

print("=== validate_file_path() ===")
for candidate in ["notes.txt", "../../etc/passwd", "file; rm -rf /", "/etc/hosts", "reports/q4.csv"]:
    try:
        print(f"accepted  {candidate!r:22s} -> {validate_file_path(candidate)}")
    except ValueError as e:
        print(f"rejected  {candidate!r:22s} -> {e}")

### A server with authorization, validation, and retries

The next cell combines the chapter's two `call_tool()` listings. The role-based gate is the outer `call_tool()`; it authenticates the token, checks the tool's required roles, and only then calls `_execute_tool()`, which is the chapter's error-handling listing: validate early, return specific messages for validation errors, and log but hide unexpected errors. `execute_query_with_retry()` supplies the retry logic the chapter describes but does not show, against an in-memory SQLite table; its first attempt fails on purpose so the retry is visible in the log.

In [ ]:
from mcp.server import Server
from mcp.types import Tool, TextContent
from mcp.shared.memory import create_connected_server_and_client_session

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("secure-server")

# An in-memory database standing in for the production backend
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE sales(month TEXT, revenue INTEGER)")
db.executemany("INSERT INTO sales VALUES (?, ?)",
               [("2024-10", 125000), ("2024-11", 131000), ("2024-12", 142000)])

_attempts = {"count": 0}

async def execute_query_with_retry(query: str, max_retries: int = 3) -> str:
    """Retry a query on transient errors with exponential backoff."""
    for attempt in range(max_retries):
        try:
            _attempts["count"] += 1
            if _attempts["count"] == 1:
                raise ConnectionError("simulated transient connection reset")
            return json.dumps(db.execute(query).fetchall())
        except ConnectionError as e:
            if attempt < max_retries - 1:
                logger.warning(f"Transient error, retry {attempt + 1}/{max_retries}: {e}")
                await asyncio.sleep(0.05 * 2 ** attempt)  # Exponential backoff
            else:
                raise


# Who holds which roles, and which roles each tool requires
_tokens = {"analyst-token": ["analyst"], "admin-token": ["analyst", "admin"]}
tool_permissions = {"database_query": ["analyst"], "read_file": ["analyst"], "delete_file": ["admin"]}

def _authenticate(token: str | None) -> list[str]:
    """Return the caller's roles. In production, validate a JWT or API key here."""
    return _tokens.get(token or "", [])


server = Server("secure-server")

@server.list_tools()
async def list_tools() -> list[Tool]:
    auth = {"_auth_token": {"type": "string", "description": "Caller's access token"}}
    return [
        Tool(name="database_query", description="Run a read-only SQL query against the sales table",
             inputSchema={"type": "object", "properties": {"query": {"type": "string"}, **auth}, "required": ["query"]}),
        Tool(name="read_file", description="Read a document by its path relative to the documents folder",
             inputSchema={"type": "object", "properties": {"path": {"type": "string"}, **auth}, "required": ["path"]}),
        Tool(name="delete_file", description="Delete a document (admin only)",
             inputSchema={"type": "object", "properties": {"path": {"type": "string"}, **auth}, "required": ["path"]}),
    ]


async def _execute_tool(name: str, arguments: dict) -> list[TextContent]:
    """The chapter's error-handling listing: validate, categorize, never leak."""
    try:
        # Validate inputs
        if not isinstance(arguments, dict):
            raise ValueError("Arguments must be a dictionary")

        if name == "database_query":
            query = arguments.get("query")
            if not query:
                raise ValueError("Query parameter is required")

            result = await execute_query_with_retry(query)
            return [TextContent(type="text", text=result)]

        elif name == "read_file":
            path = validate_file_path(arguments.get("path", ""))
            return [TextContent(type="text", text=Path(path).read_text())]

        elif name == "delete_file":
            return [TextContent(type="text", text=f"Deleted {arguments.get('path')} (simulated)")]

        else:
            logger.warning(f"Unknown tool: {name}")
            return [TextContent(type="text", text=f"Error: Unknown tool '{name}'")]

    except ValueError as e:
        # Validation errors - return specific error message
        logger.error(f"Validation error in {name}: {str(e)}")
        return [TextContent(type="text", text=f"Validation error: {str(e)}")]

    except Exception as e:
        # Unexpected errors - log details but return generic message
        logger.exception(f"Unexpected error in {name}")
        return [TextContent(type="text", text="An unexpected error occurred")]


@server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    # Verify authentication token
    user_roles = _authenticate(arguments.get("_auth_token"))
    if not user_roles:
        return [TextContent(
            type="text",
            text="Error: Authentication required"
        )]

    # Check authorization for this specific tool
    required_roles = tool_permissions.get(name, [])
    if not any(role in required_roles for role in user_roles):
        return [TextContent(
            type="text",
            text="Error: Insufficient permissions"
        )]

    # Execute the tool
    return await _execute_tool(name, arguments)


print("Secure server defined with authentication, authorization, validation, and retries")

### Exercising the server through an in-memory session

Each call below hits a different layer. The first succeeds after one retry, the second has no token, the third has the wrong role, the fourth fails path validation, the fifth reads a real document, and the sixth fails input validation.

In [ ]:
cases = [
    ("database_query", {"query": "SELECT * FROM sales", "_auth_token": "analyst-token"}),
    ("database_query", {"query": "SELECT 1"}),
    ("delete_file",    {"path": "notes.txt", "_auth_token": "analyst-token"}),
    ("read_file",      {"path": "../../etc/passwd", "_auth_token": "analyst-token"}),
    ("read_file",      {"path": "notes.txt", "_auth_token": "analyst-token"}),
    ("database_query", {"query": "", "_auth_token": "admin-token"}),
]

print("=== In-memory session against the secure server ===")
async with create_connected_server_and_client_session(server) as session:
    for name, arguments in cases:
        result = await session.call_tool(name, arguments)
        shown = {k: v for k, v in arguments.items() if k != "_auth_token"}
        print(f"\n{name}({json.dumps(shown)}) as {arguments.get('_auth_token', 'no token')}")
        print(f"  -> {result.content[0].text.strip()[:90]}")

### Connection pooling

The chapter's `DatabaseServer` keeps an `asyncpg` pool so each request borrows a connection instead of opening one. The listing below carries the one correction from the chapter edits: the constructor stores the URL it receives. It needs a PostgreSQL instance to run, so the cell only exercises it when a `DATABASE_URL` environment variable is set and otherwise reports that it was skipped.

In [ ]:
from mcp.server import Server
import asyncpg

class DatabaseServer:
    def __init__(self, database_url: str):
        self.server = Server("database-server")
        self.database_url = database_url
        self.pool = None

    async def initialize(self):
        """Create connection pool on startup"""
        self.pool = await asyncpg.create_pool(
            self.database_url,
            min_size=5,    # Keep 5 connections ready
            max_size=20    # Allow up to 20 concurrent connections
        )

    async def execute_query(self, query: str) -> list[dict]:
        async with self.pool.acquire() as connection:
            rows = await connection.fetch(query)
            return [dict(row) for row in rows]

    async def cleanup(self):
        """Close all connections on shutdown"""
        if self.pool:
            await self.pool.close()


# --- Run it (only when a PostgreSQL URL is available) ---

if os.getenv("DATABASE_URL"):
    db_server = DatabaseServer(os.environ["DATABASE_URL"])
    await db_server.initialize()
    print(await db_server.execute_query("SELECT now() AS server_time"))
    await db_server.cleanup()
else:
    print("DATABASE_URL not set; connection pooling example defined but not run")

## Part 6: Integrating MCP with an Agent

The chapter's `MCPAgent` connects a model to an MCP server and runs the thought, action, observation loop from Chapter 3 with MCP supplying the tools. This version uses the OpenAI Responses API with `gpt-5.6-luna`; the chapter prints the Anthropic Messages API version, and the structure is the same. Three details carry over from the chapter edits: `list_tools()` returns a result object whose `tools` field holds the list, the transport and session are kept on an `AsyncExitStack` for the life of the agent, and every turn is appended to one history so the model sees its earlier tool results.

The MCP tool schema maps directly onto a Responses API function tool: `name` and `description` are copied, and `inputSchema` becomes `parameters`.

In [ ]:
from openai import OpenAI

MODEL = "gpt-5.6-luna"
SYSTEM = "You are a helpful assistant. Use the tools to look things up before you answer, and keep answers to one or two sentences."


class MCPAgent:
    def __init__(self):
        self.client = OpenAI()
        self.mcp_session: Optional[ClientSession] = None
        self._stack = AsyncExitStack()

    async def initialize_mcp(self, server_command: str, server_args: list[str]):
        """Connect to MCP server"""
        server_params = StdioServerParameters(
            command=server_command,
            args=server_args
        )
        # Keep the transport and the session open for the life of the agent
        read, write = await self._stack.enter_async_context(stdio_client(server_params))
        self.mcp_session = await self._stack.enter_async_context(ClientSession(read, write))
        await self.mcp_session.initialize()

    async def close(self):
        """Shut the session and the server subprocess down"""
        await self._stack.aclose()

    async def run(self, user_message: str) -> str:
        # Get available tools from MCP server
        tools = (await self.mcp_session.list_tools()).tools

        # Convert MCP tools to the Responses API function-tool format
        openai_tools = [
            {
                "type": "function",
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.inputSchema,
            }
            for tool in tools
        ]

        history = [{"role": "user", "content": user_message}]
        while True:
            response = self.client.responses.create(
                model=MODEL, instructions=SYSTEM, tools=openai_tools,
                input=history, store=False,
                include=["reasoning.encrypted_content"],
            )
            history += response.output
            calls = [item for item in response.output if item.type == "function_call"]
            if not calls:
                return response.output_text

            # Execute every requested tool via MCP and feed the results back
            for call in calls:
                result = await self.mcp_session.call_tool(
                    name=call.name,
                    arguments=json.loads(call.arguments)
                )
                print(f"  [tool] {call.name}({call.arguments}) -> {result.content[0].text[:60]!r}")
                history.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": result.content[0].text,
                })


# --- Run it ---

agent = MCPAgent()
await agent.initialize_mcp(PYTHON, ["tool_server.py", str(DOCS)])
try:
    for question in [
        "What was our revenue in November 2024 according to the documents?",
        "Which MCP specification revision is the stable release? Search the web.",
    ]:
        print(f"=== {question} ===")
        print(await agent.run(question))
        print()
finally:
    await agent.close()

## Part 7: Framework Integration with LangChain

LangChain's official MCP adapters package, `langchain-mcp-adapters`, connects to one or more MCP servers, converts their tools to LangChain tools, and hands them to `create_agent()`, the same function Chapter 1 used. The `MultiServerMCPClient` takes a dictionary of connections keyed by server name, with the transport spelled out, so the stdio server from Part 3 and the HTTP server from Part 4 could sit side by side in one agent.

`gpt-5.6-luna` rejects function tools on the Chat Completions API unless reasoning is switched off for the call, so the model is created with `reasoning_effort="none"` as in Chapter 1.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

# One entry per MCP server; the transport is part of the connection
mcp_client = MultiServerMCPClient({
    "utility": {
        "transport": "stdio",
        "command": PYTHON,
        "args": ["tool_server.py", str(DOCS)],
    },
})

# MCP tools arrive as LangChain tools
tools = await mcp_client.get_tools()
print("=== MCP tools as LangChain tools ===")
for tool in tools:
    print(f"{tool.name}: {tool.description[:70]}...")

# Create the agent with the tools acquired from the MCP server
llm = ChatOpenAI(model=MODEL, reasoning_effort="none")
agent = create_agent(llm, tools)

# Run the agent
result = await agent.ainvoke({
    "messages": [{"role": "user", "content": "What does the handbook say about release reviews? Search the documents for the word Thursday."}]
})

print("\n=== Conversation ===")
for message in result["messages"]:
    text = message.content if isinstance(message.content, str) else json.dumps(message.content)
    print(f"{type(message).__name__:13s} {text[:100]}")

## Summary

In this notebook, we implemented:

1. **Resource Server**: A stdio server exposing workspace files, launched by the client as a subprocess and shut down when the session closed
2. **Connection Lifecycle**: The initialize handshake with its negotiated protocol version and capabilities, then `resources/list` and `resources/read`
3. **Prompt Server**: A `code-review` template rendered on request, with the handler returning the `GetPromptResult` the SDK requires
4. **Tool Server**: The chapter's `web_search` tool plus a document search, and a `CallToolResult` unpacked on the client side
5. **Streamable HTTP**: The same tools served by a background process on localhost, with the session id from the handshake
6. **Production Patterns**: Path validation, role-based access control, and retries with exponential backoff, exercised through an in-memory session
7. **Agent Integration**: A Responses API loop that discovers MCP tools, converts their schemas, and feeds results back turn by turn
8. **Framework Integration**: The same tools driving a LangChain `create_agent()` agent through the official MCP adapters

### Key Takeaways:

- An MCP server is a program; over stdio the client launches it, over HTTP it runs on its own and hands each client a session id
- The three primitives differ in who decides: the application reads resources, the user picks prompts, and the model calls tools
- Every client call returns a result object, and the fields on it (`resources`, `messages`, `content`, `isError`) are what a harness inspects
- Validation, authorization, and retries live in the server's `call_tool()` path, in that order
- A model needs only a schema translation to use MCP tools, and a framework needs only an adapter

### Next Steps:

- Replace the fixed `perform_web_search()` with a real search API and the SQLite table with the pooled PostgreSQL server
- Put the HTTP server behind TLS and the OAuth 2.1 authorization the specification defines for HTTP transports
- Move on to Chapter 6, which covers MCP in practice: security, performance, and deployment